In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from vizopt import jaxopt

In [ ]:
random_labels = ["apple", "double espresso", "quiet room", "blue", "green", "yellow"]
n_labels = len(random_labels)
point_positions = 10*np.random.rand(n_labels, 2)
label_df = pd.DataFrame({"label": random_labels, "x": point_positions[:, 0], "y": point_positions[:, 1]})
label_df["dy"] = 1.
label_df["dx"] = label_df["label"].str.len()*0.5
label_df

In [ ]:
input_parameters = {
    "point_positions": point_positions,
    "rectangle_sizes": label_df[["dx", "dy"]].values,
}
input_parameters

In [ ]:
optim_vars = {"rectangle_positions": point_positions}
optim_vars

In [ ]:
def plot_rectangles(optim_vars, input_parameters):
    _, ax = plt.subplots(figsize=(4, 3))
    n_rect = optim_vars["rectangle_positions"].shape[0]
    for i_rect in range(n_rect):
        ax.add_patch(plt.Rectangle(
            optim_vars["rectangle_positions"][i_rect, :],
            input_parameters["rectangle_sizes"][i_rect, 0],
            input_parameters["rectangle_sizes"][i_rect, 1],
            alpha=0.5
        ))
        link_x = [optim_vars["rectangle_positions"][i_rect, 0], input_parameters["point_positions"][i_rect, 0]]
        link_y = [optim_vars["rectangle_positions"][i_rect, 1], input_parameters["point_positions"][i_rect, 1]]
        ax.plot(link_x, link_y)
        
    ax.scatter(input_parameters["point_positions"][:, 0], input_parameters["point_positions"][:, 1])

plot_rectangles(optim_vars, input_parameters)

In [ ]:
from jax import numpy as jnp


def multiple_bbox_intersections(bbox_matrix, other_bbox_matrix):
    """Calculate the pairwise intersections of two sets of bounding boxes

    This vectorized implementation is more efficient than the avoided double for loop

    Args:
        bbox_matrix: numpy array of shape (n, 2, 2)
            dimensions: points, min and max, xy coordinates
        other_bbox_matrix: numpy array of shape (m, 2, 2)
            dimensions: points, min and max, xy coordinates

    Returns:
        numpy array of shape (n, m)
    """
    rep_bbox_matrix = jnp.repeat(bbox_matrix, other_bbox_matrix.shape[0], axis=0)
    rep_other_bbox_matrix = jnp.tile(other_bbox_matrix, (bbox_matrix.shape[0], 1, 1))

    coord_max = rep_bbox_matrix[:, 1, :]
    coord_min = rep_bbox_matrix[:, 0, :]
    coord_max_other = rep_other_bbox_matrix[:, 1, :]
    coord_min_other = rep_other_bbox_matrix[:, 0, :]
    intersects = jnp.clip(
        coord_max - coord_min_other, a_max=coord_max_other - coord_min
    )
    x_intersect = jnp.clip(intersects[:, 0], 0, np.inf)
    y_intersect = jnp.clip(intersects[:, 1], 0, np.inf)
    intersect_prods = x_intersect * y_intersect
    return intersect_prods.reshape(bbox_matrix.shape[0], -1)

# pairwise intersections between label bounding boxes
def calculate_intersection_loss(optim_vars, input_parameters):
    bbox_matrix = jnp.stack([
        optim_vars["rectangle_positions"],
        optim_vars["rectangle_positions"] + input_parameters["rectangle_sizes"]
    ], axis=1)

    interlabel_inters_matrix = multiple_bbox_intersections(bbox_matrix, bbox_matrix)
    interlabel_inters_array = interlabel_inters_matrix[np.triu_indices(interlabel_inters_matrix.shape[0], 1)]
    return jnp.sum(interlabel_inters_array)

calculate_intersection_loss(optim_vars, input_parameters)

In [ ]:
# distances between points and the respective labels
def calculate_point_label_distances(optim_vars, input_parameters):
    return jnp.sum(
        (optim_vars["rectangle_positions"] - input_parameters["point_positions"]) ** 2
    )


def fun_to_minimize(optim_vars):
    return 5*calculate_intersection_loss(
        optim_vars, input_parameters
    ) + calculate_point_label_distances(optim_vars, input_parameters)


optim_vars_opt, _ = jaxopt.optimize_gradient_descent(optim_vars, fun_to_minimize=fun_to_minimize)
optim_vars_opt

In [ ]:
plot_rectangles(optim_vars_opt, input_parameters)

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class ObjectiveTerm:
    name: str
    compute: Callable